# OmniMedVQA-V2 EDA

## Environment setup

Run the optional installation command in Colab only if the required packages are missing.

In [ ]:
# Directory
from pathlib import Path

def find_project_root(start=Path.cwd(), project_name="attack-on-medvqa"):
    start = Path(start).resolve()

    for path in [start, *start.parents]:
        if path.name == project_name:
            return path

    raise FileNotFoundError(f"Could not find project root named {project_name}")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "OmniMedVQA"
DATA_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = str(DATA_DIR)

print("Project root:", PROJECT_ROOT)
print("Dataset cache:", CACHE_DIR)

In [ ]:
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display
from pathlib import Path
import random

In [ ]:
OMNIMEDVQA_V2_DATASET = "mtybilly/OmniMedVQA-V2"

SUBSETS = [
    "mod-ct",
    "mod-derm",
    "mod-fundus",
    "mod-micro",
    "mod-mri",
    "mod-oct",
    "mod-us",
    "mod-xray",
    
    "qt-ai",
    "qt-dd",
    "qt-lg",
    "qt-mr",
    "qt-oba",
]


## Load OmniMedVQA-V2 test data


In [ ]:
test_datasets = []

for subset in SUBSETS:
    print(f"Loading subset: {subset}")

    test_files = f"hf://datasets/{OMNIMEDVQA_V2_DATASET}/{subset}/test-*.parquet"

    ds = load_dataset(
        "parquet",
        data_files={"test": test_files},
        split="test",
        cache_dir=CACHE_DIR,
    )

    ds = ds.add_column("subset", [subset] * len(ds))

    category = "modality" if subset.startswith("mod-") else "question_type"
    ds = ds.add_column("category", [category] * len(ds))

    test_datasets.append(ds)

test_ds = concatenate_datasets(test_datasets)

print(test_ds)
print(f"Rows: {test_ds.num_rows:,}")
print(f"Columns: {test_ds.num_columns}")

for k, v in test_ds.features.items():
    print(f"{k}: {v}")

In [ ]:
display(test_ds[0])

## Preprocess

In [ ]:
df = test_ds.remove_columns(["image"]).to_pandas()

In [ ]:
df.head()

In [ ]:
df.describe(include="all")

In [ ]:
df["modality"].value_counts()

In [ ]:
modality_mapping = {
    "MR (Mag-netic Resonance Imaging)": "MRI",
    "MR (Magnetic Resonance Imaging)": "MRI",
    "CT(Computed Tomography)": "CT",
    "CT (Computed Tomography)": "CT",
    "X-Ray": "X-ray",
    "x-ray": "X-ray",
    "ultrasound": "Ultrasound",
    "OCT (Optical Coherence Tomography": "OCT",
    "OCT (Optical Coherence Tomography)": "OCT",
    "Fundus Photography": "Fundus",
    "Microscopy Images": "Microscopy",
}

df["modality"] = df["modality"].str.strip().replace(modality_mapping)

In [ ]:
df["modality"].value_counts()

## Modality and question-type categories

In [ ]:
df_mod = df[df["category"] == "modality"].copy()
df_qt = df[df["category"] == "question_type"].copy()

print(f"Rows in modality category: {len(df_mod):,}")
print(f"Rows in question-type category: {len(df_qt):,}")

In [ ]:
modality_subset_counts = (
    df_mod.groupby(["subset", "modality"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

modality_subset_counts["percent"] = modality_subset_counts["count"] / modality_subset_counts["count"].sum() * 100

display(modality_subset_counts)

plt.figure(figsize=(12,7))

ax = sns.barplot(
    data=modality_subset_counts,
    x="count",
    y="modality"
)

for i, row in modality_subset_counts.iterrows():
    ax.text(
        row["count"],
        i,
        f' {row["percent"]:.1f}%',
        va="center"
    )

plt.title("Number of Test Samples per Modality Subset")
plt.xlabel("Number of samples")
plt.ylabel("Modality")
plt.tight_layout()
plt.show()

In [ ]:
qt_subset_counts = (
    df_qt.groupby(["subset", "question_type"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

qt_subset_counts["percent"] = qt_subset_counts["count"] / qt_subset_counts["count"].sum() * 100

display(qt_subset_counts)

plt.figure(figsize=(13,7))

ax = sns.barplot(
    data=qt_subset_counts,
    x="count",
    y="question_type"
)


for i, row in qt_subset_counts.iterrows():
    ax.text(
        row["count"],
        i,
        f' {row["percent"]:.1f}%',
        va="center"
    )

plt.title("Number of Test Samples per Question-Type Subset")
plt.xlabel("Number of samples")
plt.ylabel("Question type")
plt.tight_layout()
plt.show()

# Answer Distribution

In [ ]:
display(df_mod["answer_letter"].value_counts())

answer_counts = df_mod["answer_letter"].value_counts().reindex(["A", "B", "C", "D"])

plt.figure(figsize=(4, 4))

plt.pie(
    answer_counts,
    labels=answer_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Distribution of Correct Answer Letters")
plt.tight_layout()
plt.show()

In [ ]:
display(df_qt["answer_letter"].value_counts())

answer_counts = df_qt["answer_letter"].value_counts().reindex(["A", "B", "C", "D"])

plt.figure(figsize=(4, 4))

plt.pie(
    answer_counts,
    labels=answer_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Distribution of Correct Answer Letters")
plt.tight_layout()
plt.show()

## Visualize random example

In [ ]:
def show_random_examples(dataset, n=3, seed=42):

    rng = random.Random(seed)
    sample_size = min(n, len(dataset))
    indices = rng.sample(range(len(dataset)), sample_size)

    fields_to_print = dataset.column_names

    for display_index, dataset_index in enumerate(indices, start=1):
        sample = dataset[dataset_index]
        print("=" * 100)
        print(f"Example {display_index} | dataset index: {dataset_index}")

        display(sample["image"])
        print(f"modality: {sample['modality']}")

        for field in fields_to_print:
            value = sample[field]
            print(f"{field}: {value}")

show_random_examples(test_ds, n=3, seed=42)

## One sample from each modality

The first modality-category sample is shown for each normalized modality.

In [ ]:
# Keep the image row aligned with the corresponding row in test_ds.
modality_samples = (
    df[df["category"] == "modality"]
    .assign(dataset_index=lambda frame: frame.index)
    .sort_values("dataset_index")
    .drop_duplicates("modality")
    .sort_values("modality")
)

n_samples = len(modality_samples)
n_cols = 4
n_rows = (n_samples + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4.5 * n_rows))
axes = np.atleast_1d(axes).ravel()

for ax, (_, row) in zip(axes, modality_samples.iterrows()):
    sample = test_ds[int(row["dataset_index"])]
    ax.imshow(sample["image"])
    ax.set_title(f"{row['modality']} ({row['subset']})", fontsize=12)
    ax.set_xlabel(
        f"Q: {sample['question']}\nAnswer: {sample['answer_text']}",
        fontsize=8,
    )
    ax.axis("off")

for ax in axes[n_samples:]:
    ax.axis("off")

fig.suptitle("Representative OmniMedVQA-V2 images by modality", fontsize=16)
plt.tight_layout()
plt.show()

## Summary


In [ ]:
print(f"Total rows in test dataset: {len(test_ds):,}")
print(f"Total rows in modality category: {len(df_mod):,}")
print(f"Total rows in question-type category: {len(df_qt):,}")

display(modality_subset_counts)
display(qt_subset_counts)